In [160]:
import alerce
import pandas as pd
import pyvo as vo
from astropy.time import Time
import numpy as np

In [30]:
alerce_tap = vo.dal.TAPService('https://tap.alerce.online/tap')

In [71]:
day = int(Time.now().mjd - 11) + 0.5 #mjd
day

61161.5

In [75]:
# this query uses the lsst_detection table, maybe another will be better
query = '''
SELECT TOP 100
    oid, 
    AVG(sid) as sid,
    MAX(timeprocessedmjdtai) AS mjd,
    AVG(scienceflux) as scienceflux, 
    AVG(templateflux) AS templateflux
FROM
    alerce_tap.lsst_detection
WHERE
    timeprocessedmjdtai > ''' + str(day) + '''
    AND ABS((scienceflux - templateflux)/scienceflux) > 0.4
    AND sid = 1
GROUP BY oid
ORDER BY mjd DESC

'''

q_tot = '''
SELECT
    oid,
    timeprocessedmjdtai as mjd,
    scienceflux
FROM
    alerce_tap.lsst_detection
WHERE
    timeprocessedmjdtai > ''' + str(day) + '''
'''

In [60]:
query

'\nSELECT TOP 100\n    oid, \n    AVG(sid) as sid,\n    MAX(timeprocessedmjdtai) AS mjd,\n    AVG(scienceflux) as scienceflux, \n    AVG(templateflux) AS templateflux\nFROM\n    alerce_tap.lsst_detection\nWHERE\n    timeprocessedmjdtai > 61152.5\n    AND ABS((scienceflux - templateflux)/scienceflux) > 0.4\n    AND sid = 1\nGROUP BY oid\nORDER BY mjd DESC\n\n'

In [73]:
result = alerce_tap.search(query).to_table().to_pandas()
result

,oid,sid,mjd,scienceflux,templateflux
0,170046091660623958,1.0,61162.081129,6806.273071,3911.821655
1,170345142502817866,1.0,61162.081109,4864.913574,-86.095413
2,170028514403156077,1.0,61162.081109,-33.232351,17612.522786
3,170050496606240987,1.0,61162.081088,11634.169922,3561.272461
4,170345142506487925,1.0,61162.081084,4431.777344,305.537445
...,...,...,...,...,...
95,313963404992184401,1.0,61162.071376,32578.682129,48246.228516
96,170345139789103127,1.0,61162.071366,7159.606934,-186.982178
97,170028514667921554,1.0,61162.071363,131.624533,172166.398438
98,170345139799588877,1.0,61162.071359,6531.080078,3069.733887


In [76]:
magstats_test = alerce_tap.search(que_tot).to_table().to_pandas()
magstats_test

,oid,mjd,scienceflux
0,21163615972700232,61162.065507,23048.197266
1,21163607365597258,61162.065597,17575.861328
2,170028510687527780,61162.065597,84799.585938
3,21164728051905879,61162.065619,16120.556641
4,313853517483737142,61162.065512,21896.892578
...,...,...,...
2120,170028510642962490,61162.081109,66567.234375
2121,21163611663053904,61162.081068,30362.101562
2122,314003014207668253,61162.081129,147952.937500
2123,313853517711802444,61162.081070,25555.871094


why do these 2 querys not give at least SOME of the same oids? the lsst_detection query is far more limiting than the magstat query (due to the flux ratio requirement, inspired by the antares [highfluxchange filter](https://nsf-noirlab.gitlab.io/csdc/antares/devkit/reference/filters/highfluxchange/)), and so while it makes sense that they arent identical, i feel that the magstat filter should have MORE oids, not less (and certainly not zero). like, the first oid given has 104 detections, so why doesn't it appear in the magstat query? inch resting,,,,,,,,,,,,

but looking at that first query again, thats not what im looking for. it compares the AVG of the scienceflux to the templateflux, so it says nothing about the recent detections of the oid. herm,,,,,,,,,

i dont think alerce has an easy way to query lightcurve data by oid, which (if true) would mean it Definitely isn't the broker to use. 

In [117]:
day_beg = int(Time.now().mjd - 13) + 0.5 #mjd
day_end = day_beg + 1
q_tot = '''
SELECT
    oid,
    timeprocessedmjdtai as mjd,
    scienceflux
FROM
    alerce_tap.lsst_detection
WHERE
    timeprocessedmjdtai BETWEEN ''' + str(day_beg) + ''' AND ''' + str(day_end) + '''
'''

q_magcut = '''
SELECT
    oid,
    timeprocessedmjdtai as mjd,
    scienceflux
FROM
    alerce_tap.lsst_detection
WHERE
    timeprocessedmjdtai BETWEEN ''' + str(day_beg) + ''' AND ''' + str(day_end) + '''
    AND -2.5*log10(ABS(scienceflux))+31.4 > 21
'''

In [118]:
tot = alerce_tap.search(q_tot).to_table().to_pandas()
cut = alerce_tap.search(q_magcut).to_table().to_pandas()
print(f' for mjd {day_beg}, {round(100*len(cut)/len(tot), 2)}% of the detections (alerts) had flux > 10k (mag < 21)')

 for mjd 61159.5, 45.24% of the detections (alerts) had flux > 10k (mag < 21)


In [ ]:
today = Time.now().mjd
days_ago = 0
amts = {}
while days_ago < 20:
    day_beg = int(today - days_ago) + 0.5 #mjd
    day_end = day_beg + 1
    
    q_tot_loop = '''
    SELECT
        count(oid) as oid
    FROM
        alerce_tap.lsst_detection
    WHERE
        timeprocessedmjdtai BETWEEN ''' + str(day_beg) + ''' AND ''' + str(day_end) + '''
    '''

    q_magcut_loop = '''
    SELECT
        count(oid) as oid
    FROM
        alerce_tap.lsst_detection
    WHERE
        timeprocessedmjdtai BETWEEN ''' + str(day_beg) + ''' AND ''' + str(day_end) + '''
        AND -2.5*log10(ABS(scienceflux))+31.4 < 21
    '''

    tot_l = alerce_tap.search(q_tot_loop).to_table().to_pandas()
    cut_l = alerce_tap.search(q_magcut_loop).to_table().to_pandas()

    if tot_l['oid'][0] == 0:
        percent = 'nan'
    else:
        percent = float(cut_l['oid'][0]/tot_l['oid'][0])

    amts[day_beg] = [int(tot_l['oid'][0]), int(cut_l['oid'][0]), percent]
    
    days_ago +=1
    

In [129]:
amts

{61172.5: [0, 0, 'nan'],
 61171.5: [0, 0, 'nan'],
 61170.5: [0, 0, 'nan'],
 61169.5: [0, 0, 'nan'],
 61168.5: [0, 0, 'nan'],
 61167.5: [0, 0, 'nan'],
 61166.5: [0, 0, 'nan'],
 61165.5: [0, 0, 'nan'],
 61164.5: [0, 0, 'nan'],
 61163.5: [0, 0, 'nan'],
 61162.5: [0, 0, 'nan'],
 61161.5: [3331, 1762, 0.5289702791954368],
 61160.5: [3554, 1946, 0.5475520540236354],
 61159.5: [7239, 3964, 0.5475894460560851],
 61158.5: [0, 0, 'nan'],
 61157.5: [0, 0, 'nan'],
 61156.5: [0, 0, 'nan'],
 61155.5: [0, 0, 'nan'],
 61154.5: [0, 0, 'nan'],
 61153.5: [0, 0, 'nan']}

im unclear why its formatted this way (which i will probs need to figure out in the future), but from these at least the trend is for ~ 54% of the detections from a given night would be within the observation capabilities of SDSS (assuming limit being 21 mag for SNR 3 using 4 observation windows: 4x15min, these may be adjusted in the future)

In [152]:
def mag_perc(day, maglim):
    day_end = day + 1

    
    q_tot_loop = '''
    SELECT
        count(oid) as oid
    FROM
        alerce_tap.lsst_detection
    WHERE
        timeprocessedmjdtai BETWEEN ''' + str(day) + ''' AND ''' + str(day_end) + '''
    '''

    q_magcut_loop = '''
    SELECT
        count(oid) as oid
    FROM
        alerce_tap.lsst_detection
    WHERE
        timeprocessedmjdtai BETWEEN ''' + str(day) + ''' AND ''' + str(day_end) + '''
        AND -2.5*log10(ABS(scienceflux))+31.4 < ''' +  str(maglim)

    tot_l = alerce_tap.search(q_tot_loop).to_table().to_pandas()
    cut_l = alerce_tap.search(q_magcut_loop).to_table().to_pandas()

    if tot_l['oid'][0] == 0:
        percent = 'nan'
    else:
        percent = float(cut_l['oid'][0]/tot_l['oid'][0])

    return [day, int(tot_l['oid'][0]), int(cut_l['oid'][0]), percent]

In [173]:
# looking for amts for 50 days of observations, this skips over non-observation days
today = Time.now().mjd
days_ago = 1
day_count = 25
amts = []
while days_ago <= day_count:
    day_beg = int(today - days_ago) + 0.5 #mjd
    day_amts = mag_perc(day_beg, 18)
    if day_amts[2] == 0:
        day_count +=1
    else:
        amts.append(day_amts)
    days_ago += 1

amts_18 = np.array(amts)

In [175]:
# looking for amts for 25 days of observations, this skips over non-observation days
today = Time.now().mjd
days_ago = 1
day_count = 25
amts = []
while days_ago <= day_count:
    day_beg = int(today - days_ago) + 0.5 #mjd
    day_amts = mag_perc(day_beg, 19)
    if day_amts[2] == 0:
        day_count +=1
    else:
        amts.append(day_amts)
    days_ago += 1

amts_19 = np.array(amts)

In [177]:
# looking for amts for 50 days of observations, this skips over non-observation days
today = Time.now().mjd
days_ago = 1
day_count = 25
amts = []
while days_ago <= day_count:
    day_beg = int(today - days_ago) + 0.5 #mjd
    day_amts = mag_perc(day_beg, 20)
    if day_amts[2] == 0:
        day_count +=1
    else:
        amts.append(day_amts)
    days_ago += 1

amts_20 = np.array(amts)

In [167]:
# looking for amts for 50 days of observations, this skips over non-observation days
today = Time.now().mjd
days_ago = 1
day_count = 25
amts = []
while days_ago <= day_count:
    day_beg = int(today - days_ago) + 0.5 #mjd
    day_amts = mag_perc(day_beg, 21)
    if day_amts[2] == 0:
        day_count +=1
    else:
        amts.append(day_amts)
    days_ago += 1

amts_21 = np.array(amts)

In [187]:
magcutperc = {'18': amts_18,
              '19': amts_19,
              '20': amts_20,
              '21': amts_21,
             }
for mag in magcutperc:
    print(f'mag cut of {mag} is {round(100*np.mean(magcutperc[mag][:,3]),2)} +-{round(100*np.std(magcutperc[mag][:,3]),2)}% of total detections in a given night')
    

mag cut of 18 is 4.38 +-6.13% of total detections in a given night
mag cut of 19 is 9.85 +-7.11% of total detections in a given night
mag cut of 20 is 25.25 +-8.37% of total detections in a given night
mag cut of 21 is 48.42 +-9.39% of total detections in a given night
